In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

import kagglehub

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
competition_dir = Path(".")
if not (competition_dir / "data" / "train.csv").is_file() and Path("playground-series-s6e10/data/train.csv").is_file():
    competition_dir = Path("playground-series-s6e10")
data_dir = competition_dir / "data"
train_path = data_dir / "train.csv"
test_path = data_dir / "test.csv"

if not (train_path.is_file() and test_path.is_file()):
    kagglehub.competition_download(
        "playground-series-s6e10",
        output_dir=str(data_dir),
        force_download=True,
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print(f"Loaded {len(train):,} training rows and {len(test):,} test rows.")

Loaded 699,635 training rows and 299,844 test rows.


### Simple baseline model

Use a logistic regression pipeline: impute missing numeric values, scale numeric features, one-hot encode categorical features, and train on all labeled rows.

In [ ]:
features = train.drop(columns=["id", "satisfaction"])
test_features = test.drop(columns=["id"])
categorical_columns = features.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_columns = features.columns.difference(categorical_columns).tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_columns),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_columns),
    ]
)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=200)),
])
model.fit(features, train["satisfaction"].astype(int))
test_probabilities = model.predict_proba(test_features)[:, 1]
print("Baseline model trained; generated satisfaction probabilities for the test set.")

Baseline model trained; generated satisfaction probabilities for the test set.


In [4]:
submission = pd.read_csv(data_dir / "sample_submission.csv")
submission["satisfaction"] = test_probabilities
assert len(submission) == len(test)
assert submission["id"].equals(test["id"])

submission_path = competition_dir / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Saved {len(submission):,} predictions to {submission_path}.")
submission.head()

Saved 299,844 predictions to submission.csv.


,id,satisfaction
0,699635,0.089519
1,699636,0.582812
2,699637,0.471503
3,699638,0.137832
4,699639,0.003278


### Submit to Kaggle

Run this cell when you are ready to upload the generated file. It uses the Kaggle CLI and your configured credentials. Each run creates a real competition submission and counts toward Kaggle's submission limit.

In [14]:
import subprocess
import sys

submit = 0
if submit:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "kaggle",
            "competitions",
            "submit",
            "playground-series-s6e10",
            "--file",
            str(submission_path.resolve()),
            "--message",
            "Simple logistic regression baseline",
        ],
        check=True,
    )
    print(f"Submitted {len(submission):,} predictions to the competition.")